# sbikit quickstart

A complete simulation-based inference analysis on a toy problem, in about two minutes on a laptop CPU:

1. simulate a training set and a fiducial/derivative set,
2. compress the data vector (MOPED, and a neural VMIM compressor as the alternative),
3. learn the likelihood of the summaries with a normalising flow,
4. sample the posteriors of many test simulations in one batched MCMC ensemble,
5. check calibration (TARP, marginal coverage) and constraining power (figure of merit),
6. save the model and a record of the run.

**The toy model.** A "power spectrum" with an amplitude $A$, a tilt $n$ and a constant noise floor $N$,
measured in 30 bins of $k$ with Gaussian noise:

$$x_i = A\,(k_i / k_0)^{n} + N + 0.1\,\epsilon_i, \qquad \epsilon_i \sim \mathcal{N}(0, 1).$$

Nothing below depends on the simulator: replace `simulate` with your own forward model.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import sbikit as sk
from sbikit.compress import MOPED, VMIMCompressor
from sbikit.metrics import evaluate

device = sk.resolve_device("auto")          # "cuda" if there is a GPU, else "cpu"
rng = np.random.default_rng(0)
print(f"sbikit {sk.__version__} on {device}")

In [ ]:
# plot style: thin marks, quiet axes
INK, MUTED, GRID, BLUE, ORANGE = "#0b0b0b", "#52514e", "#e4e3df", "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 2, "font.size": 10, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "figure.dpi": 110,
})

## 1. Simulations

`Dataset` holds the parameters the flow will be conditioned on (`theta`), the data vectors (`x`) and the parameter
names. `resolve_splits` checks splits you pass, or draws random ones.

If several simulations share something that correlates them (the same N-body run seen through different
footprints, say), pass their ids as `groups=`: random splits then never separate a group, and splits you pass
yourself are checked for it.

In [ ]:
names = ["A", "n", "N"]
low, high = np.array([0.5, -1.5, 0.0]), np.array([1.5, -0.5, 0.5])
k = np.geomspace(0.1, 1.0, 30)


def simulate(theta, noise):
    # theta (M, 3), noise (M, 30) standard normal draws -> data vectors (M, 30)
    A, n, N = theta[:, 0:1], theta[:, 1:2], theta[:, 2:3]
    return A * (k / 0.3) ** n + N + 0.1 * noise


theta = rng.uniform(low, high, size=(8000, 3))
ds = sk.Dataset(theta, simulate(theta, rng.normal(size=(8000, 30))), names)
splits = sk.resolve_splits(ds, fractions={"train": 0.8, "val": 0.1, "test": 0.1}, seed=0)
tr, va, te = splits["train"], splits["val"], splits["test"]
print({name: len(idx) for name, idx in splits.items()}, "| split hash", splits.hash())

## 2. Compression

### MOPED

MOPED needs simulations at one fiducial point for the covariance, and the derivative of the mean data vector with
respect to each parameter. `FiducialSet` takes the derivatives either directly (`dmu=`) or as simulations at
$\theta_\mathrm{fid} \pm$ a step. Each $\pm$ pair must share its random seed, so that the noise cancels in the
difference; here that is the same `noise` array for both.

In [ ]:
theta_fid = np.array([1.0, -1.0, 0.25])
step = np.array([0.05, 0.05, 0.05])

x_fid = simulate(np.tile(theta_fid, (2000, 1)), rng.normal(size=(2000, 30)))
shared_noise = rng.normal(size=(200, 30))
x_minus = np.stack([simulate(np.tile(theta_fid - step * e, (200, 1)), shared_noise) for e in np.eye(3)])
x_plus = np.stack([simulate(np.tile(theta_fid + step * e, (200, 1)), shared_noise) for e in np.eye(3)])

fid = sk.FiducialSet(x_fid, names, theta_fid, x_minus, x_plus, delta=2 * step)
moped = MOPED(fid)
moped.fit()
s = moped(ds.x)
print("data vector", ds.x.shape, "-> summaries", s.shape, "for", moped.kept_params)

### A neural compressor instead

MOPED is linear and optimal only near the fiducial point. The neural compressors learn from the whole training
set: `fit(x_train, theta_train, x_val, theta_val)` returns the validation loss (without a validation set, a
tenth of the training set is held out), and calling the compressor returns summaries. `VMIMCompressor` maximises
the mutual information between summaries and parameters; `MSECompressor` and `HierarchicalVMIM` are trained the
same way, while `IMNN`, like MOPED, is trained on a fiducial set.

Either set of summaries can go into the next step. This notebook continues with MOPED.

In [ ]:
vmim = VMIMCompressor(n_out=3, config={"widths": (64, 64), "epochs": 40, "patience": 10, "lr": 2e-3},
                      device=device, seed=0)
val_loss = vmim.fit(ds.x[tr], ds.theta[tr], ds.x[va], ds.theta[va])
print(f"VMIM validation loss {val_loss:.3f} after {len(vmim.history)} epochs; summaries {vmim(ds.x[te]).shape}")

## 3. Neural likelihood estimation

`NLE` is a conditional normalising flow for $p(s \mid \theta)$. It z-scores summaries and parameters internally
and stores the scalers with the weights. `fit` stops early on the validation loss and keeps the best epoch.

In [ ]:
nle = sk.NLE(flow={"model": "maf", "hidden_features": 64, "num_transforms": 4}, device=device, seed=0)
val_loss, epochs = nle.fit(s[tr], ds.theta[tr], s[va], ds.theta[va],
                           train={"lr": 2e-3, "max_epochs": 80, "stop_after_epochs": 10})
print(f"validation loss {val_loss:.3f} after {epochs} epochs")

fig, ax = plt.subplots(figsize=(5.2, 2.8))
ax.plot(np.arange(1, epochs + 1), nle.history, color=BLUE)
ax.set(xlabel="epoch", ylabel=r"$-\log p(s \mid \theta)$", title="Validation loss of the flow")
fig.tight_layout()

## 4. Posteriors for 200 test simulations at once

`sample_posteriors` runs one ensemble sampler over all targets, each with its own walkers. The chain is extended
in blocks until every target passes the convergence checks ($\hat R$, effective sample size, autocorrelation
time), or `max_steps` is reached. Targets that did not converge are flagged in `post.converged`, with a warning.

The prior is a box, optionally with density terms (`Gaussian`, `KDE`) and support constraints (`Polygon2D`, any
callable). Walkers start in a small ball around `init_centre`: the truths for test simulations, a fiducial point
or best fit for real data.

In [ ]:
prior = sk.Prior(low, high, names=names)
test = te[:200]
config = sk.MCMCConfig(nwalkers=64, burn_in=500, block=500, max_steps=8000, min_ess=1000, n_samples=2000, seed=0)

post = sk.sample_posteriors(nle, s[test], prior, config, init_centre=ds.theta[test], init_sd=ds.theta[tr].std(0))
print(post)

In [ ]:
# the test simulation closest to the fiducial point (in units of the prior width)
i = int(np.argmin((((ds.theta[test] - theta_fid) / (high - low)) ** 2).sum(1)))
labels = [r"amplitude $A$", r"tilt $n$", r"noise floor $N$"]
fig, axes = plt.subplots(1, 3, figsize=(9, 2.7))
for p, ax in enumerate(axes):
    ax.hist(post.samples[i, :, p], bins=40, color=BLUE, alpha=0.85, edgecolor="#fcfcfb", linewidth=0.4)
    ax.axvline(ds.theta[test[i], p], color=INK, linewidth=1.5)
    ax.set(xlabel=labels[p], yticks=[])
    ax.grid(False)
    ax.spines["left"].set_visible(False)
axes[0].annotate("truth", (ds.theta[test[i], 0], 1.0), xycoords=("data", "axes fraction"), xytext=(-5, -2),
                 textcoords="offset points", va="top", ha="right", color=MUTED)
axes[0].set_title("Posterior of one test simulation, with the prior range on the x-axis")
for p, ax in enumerate(axes):
    ax.set_xlim(low[p], high[p])
fig.tight_layout()

## 5. Is it calibrated, and how constraining is it?

`evaluate` returns one standard record for posteriors of simulations with known truths:

- **TARP** (Lemos et al. 2023) on all parameters and on any named subset, summarised by two scalars with bootstrap
  errors: `bias` (negative: the coverage curve lies below the diagonal) and `confidence` (negative: overconfident);
- **marginal coverage** of the central 68 % and 95 % intervals per parameter;
- the **figure of merit** $-\tfrac12 \log\det \mathrm{Cov}$ of named parameter sets;
- z-scores of the posterior means and median posterior widths. A systematic shift of the posteriors shows up in
  `z_mean`; the TARP `bias` describes the coverage curve, not such a shift.

The record is a plain dict, ready for `json.dump`; the arrays hold the curves and per-simulation values.

In [ ]:
record, arrays = evaluate(post.samples, ds.theta[test], names,
                          fom_sets={"A_n": ["A", "n"]}, tarp_sets={"A_n": ["A", "n"]}, device=device)

for key, t in record["tarp"].items():
    print(f"TARP {key:>4}: bias {t['bias']:+.3f} +- {t['bias_sd']:.3f}   "
          f"confidence {t['confidence']:+.3f} +- {t['confidence_sd']:.3f}")
for name, m in record["marginal"].items():
    print(f"coverage {name}: 68% -> {m['cov68']:.2f}   95% -> {m['cov95']:.2f}")
print(f"mean log FoM (A, n): {record['fom']['A_n']['log_fom_mean']:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11.5, 2.9), gridspec_kw={"width_ratios": [1.15, 1, 1, 1]})

ax = axes[0]
ax.plot([0, 1], [0, 1], color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
ax.plot(arrays["tarp_alpha"], arrays["tarp_ecp_all"], color=BLUE)
ax.annotate("calibrated", (0.62, 0.62), xytext=(6, -12), textcoords="offset points", color=MUTED)
ax.text(0.06, 0.94, "posteriors,\nall parameters", transform=ax.transAxes, va="top", color=INK)
ax.set(xlabel="credibility level", ylabel="expected coverage", title="TARP", xlim=(0, 1), ylim=(0, 1))
ax.set_aspect("equal")

for p, ax in enumerate(axes[1:]):
    truth, mean, sd = ds.theta[test, p], arrays["post_mean"][:, p], arrays["post_std"][:, p]
    ax.errorbar(truth, mean, yerr=sd, fmt="none", ecolor=BLUE, elinewidth=0.6, alpha=0.35)
    ax.scatter(truth, mean, s=9, color=BLUE, edgecolor="#fcfcfb", linewidth=0.3, zorder=3)
    ax.plot([low[p], high[p]], [low[p], high[p]], color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
    ax.set(xlabel=f"true {names[p]}", title=labels[p])
axes[1].set_ylabel("posterior mean ± sd")
fig.tight_layout()

## 6. Save the model and record the run

Checkpoints hold plain tensors and settings (they load with `weights_only=True`), together with the package
versions. `run_record` writes what is needed to trace a result: configuration, versions, git hashes, the hash of
the splits and the device.

In [ ]:
out = Path(tempfile.mkdtemp())
sk.save(nle, out / "nle.pt")
sk.save(moped, out / "moped.pt")
sk.save(prior, out / "prior.pt")
sk.run_record(out / "run.json", config={"flow": nle.flow_config, "mcmc": config}, splits=splits,
              extra={"metrics": record})

nle2, moped2 = sk.load(out / "nle.pt", device), sk.load(out / "moped.pt")
same = torch.allclose(nle2.log_likelihood(moped2(ds.x[te[:5]]), ds.theta[te[:5]]),
                      nle.log_likelihood(s[te[:5]], ds.theta[te[:5]]), atol=1e-4)
print(sorted(p.name for p in out.iterdir()), "| reloaded model gives the same likelihood:", same)

## Where to go next

- **Real data:** `sk.sample_posteriors(nle, moped(x_obs), prior, init_centre=theta_fid)` samples a single
  observation; `sbikit.blinding` shifts the samples by a hidden offset until the analysis is frozen.
- **Hyperparameters:** `sbikit.hpt.tune_nle` searches flow architectures with Optuna, by validation loss or by the
  figure of merit subject to TARP constraints; `tune_compressor` does the same for compressors.
- **Other parameters than the flow was trained on:** pass `transform=` to `sample_posteriors`
  (`sbikit.transforms`).
- **Several flows:** `sk.Ensemble([nle_1, nle_2, ...])` averages their likelihoods and is used like an `NLE`.
- **Less typing:** every setting used above has a default. `sk.NLE().fit(s, theta)` followed by
  `sk.sample_posteriors(nle, s_obs, prior)` is a complete run.
- **Your own parts:** flows, compression networks, scalers, prior terms and MCMC moves can be replaced by your own;
  see "Extending sbikit" in the README.

Every class and function has a docstring: `help(sk.sample_posteriors)`.